# TweetAge v7: ablacion de normalizacion con grupos en minusculas

La particion de v6 agrupa textos normalizados conservando mayusculas. En una comprobacion posterior, el primer pliegue tenia 61 textos compartidos al pasar a minusculas (162 filas de validacion afectadas). Esta ablacion crea una particion distinta por normalizacion A en minusculas y comprueba cero coincidencias para cada brazo, en cada pliegue y entre desarrollo y holdout. No cambia ni utiliza los indices de v6 para entrenar; solo verifica el NPZ anterior y consulta sus tamanos como referencia.

Se conserva el modelo de v6: ModifiedKN por clase, caracteres de orden 4, palabras de orden 3, lambda=0.75, score bruto mas log prior y sin calibracion. A se recalcula en esta corrida. B elimina solo el limite de repeticiones. C cambia solo el paso a minusculas en el nivel de palabras; caracteres ya conserva mayusculas. D se omite porque v6 ya conserva emojis y hashtags.

El holdout se estratifica por grupos, usando clase mayoritaria con desempate por orden de edad. Se elige la fraccion de grupos de una rejilla fija de 0.180 a 0.260, paso 0.001, exclusivamente por cercania al numero de filas de holdout de v6; no se consultan metricas ni se entrena para esa eleccion. Un grupo nunca se reparte. Si los tamaños exactos no pueden coincidir, se informa la diferencia.

Regla fijada: B o C se acepta solo si la media de las cinco diferencias pareadas de F1 macro respecto a A es positiva y mayor que su desviacion estandar muestral (ddof=1). Entre aceptados se elige la mayor media, con desempate alfabetico. Elegir entre brazos introduce optimismo de seleccion. Si ninguno se acepta, se omiten holdout y CSV.

Los valores historicos permitidos son v5 calibrado=0.43847, v6 sin calibrar=0.44106 y v6 calibrado=0.44353; no son el baseline de esta ablacion. El holdout de v6 pudo compartir textos en minusculas, por lo que no se compara directamente con el nuevo. Los scores publicos de Kaggle tampoco son comparables con holdout.

In [1]:
import ast
import csv
import gc
import hashlib
import html
import json
import math
import platform
import re
import time
from collections import Counter
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from IPython.display import display, Markdown

SEED = 42
np.random.seed(SEED)
CLASSES = np.array(['13-17','18-24','25-34','35-49','50-64','65-xx'])
CHAR_ORDER,WORD_ORDER,WEIGHT = 4,3,0.75
BASE_EPSILON = 1e-6
BOS,EOS,UNK = '\x02','\x03','\x01'
roots = [Path.cwd(),Path.cwd()/'workshops'/'Kaggle-tweet-age',Path.cwd()/'natural-language-processing-workshops'/'workshops'/'Kaggle-tweet-age']
OUT_DIR = next((candidate for candidate in roots if (candidate/'TweetAge_v7_normalizacion_ablacion.ipynb').is_file()),None)
assert OUT_DIR is not None,'Ejecute desde la carpeta del notebook o la raiz del repo.'
NOTEBOOK_PATH = OUT_DIR/'TweetAge_v7_normalizacion_ablacion.ipynb'
SPLIT_PATH = OUT_DIR/'split_idx_v7_lower_seed42.npz'
SUB_PATH = OUT_DIR/'age_submission_v7_normalizacion.csv'
DATA_DIR = OUT_DIR/'tweet-age-2026-puj'
assert not SPLIT_PATH.exists() and not SUB_PATH.exists(),'Existe una salida nueva; detener sin sobrescribir.'
INPUT_HASHES = {name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in ['age_train.csv','age_test.csv','age_submission.csv']}
PROTECTED_HASHES = {name:hashlib.sha256((OUT_DIR/name).read_bytes()).hexdigest() for name in ['TweetAge_v5_charLM_kn.ipynb','TweetAge_v6_charLM_kn.ipynb','TweetAge_v7_ordinal_kn.ipynb','TweetAge_v7_descuento_concentracion.ipynb','split_idx_v6_seed42.npz','requirements.txt']}
assert PROTECTED_HASHES['split_idx_v6_seed42.npz']=='fb299a3ef97a2a6948685163ba80aef5705c442e9ccda66c2c1fb0bce1e43c4d','NPZ v6 distinto al verificado; detener.'
with np.load(OUT_DIR/'split_idx_v6_seed42.npz',allow_pickle=False) as old_split:
    assert str(old_split['train_sha256'])==INPUT_HASHES['age_train.csv']
    v6_development_rows = len(old_split['train_idx'])
    v6_holdout_rows = len(old_split['valid_idx'])
print('Python',platform.python_version(),'numpy',np.__version__,'pandas',pd.__version__,'sklearn',sklearn.__version__)
print('Semilla',SEED,'NPZ v6 verificado SOLO LECTURA; no se utilizaran sus indices.')
print('SHA-256 NPZ v6:',PROTECTED_HASHES['split_idx_v6_seed42.npz'])
def check_compliance(notebook):
    forbidden = {'tfidfvectorizer', 'countvectorizer', 'hashingvectorizer', 'dictvectorizer', 'feature_extraction', 'linear_model', 'transformers', 'transformer', 'embedding', 'embeddings', 'word2vec', 'glove'}
    problems = []
    for cell_number, cell in enumerate(notebook['cells'], 1):
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
        for node in ast.walk(ast.parse(source)):
            names = []
            if isinstance(node, ast.Name):
                names = [node.id]
            elif isinstance(node, ast.Attribute):
                names = [node.attr]
            elif isinstance(node, ast.Import):
                names = [alias.name for alias in node.names]
            elif isinstance(node, ast.ImportFrom):
                names = [node.module or ''] + [alias.name for alias in node.names]
            for name in names:
                if any(part.lower() in forbidden for part in name.split('.')):
                    problems.append((cell_number, node.lineno, name))
            if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in {'eval', 'exec', '__import__'}:
                problems.append((cell_number, node.lineno, 'ejecucion dinamica'))
    if problems:
        raise RuntimeError(f'Guardia AST: codigo prohibido {problems}')
    return True

with NOTEBOOK_PATH.open(encoding='utf-8') as handle:
    assert check_compliance(json.load(handle))
print('Guardia AST de v6 superada antes de entrenar.')


Python 3.12.14 numpy 2.5.4 pandas 3.0.6 sklearn 1.9.1
Semilla 42 NPZ v6 verificado SOLO LECTURA; no se utilizaran sus indices.
SHA-256 NPZ v6: fb299a3ef97a2a6948685163ba80aef5705c442e9ccda66c2c1fb0bce1e43c4d
Guardia AST de v6 superada antes de entrenar.


In [2]:
def verify_csv(path, columns, delimiter, bom):
    with path.open('rb') as handle:
        prefix = handle.read(32768)
    first_line = prefix.decode('utf-8-sig').splitlines()[0]
    actual = csv.Sniffer().sniff(first_line, delimiters=',;\t|').delimiter
    header = next(csv.reader([first_line], delimiter=actual))
    assert actual == delimiter and prefix.startswith(b'\xef\xbb\xbf') == bom, (path.name, actual)
    assert header == columns, (path.name, header)
    print(path.name, 'BOM=', bom, 'separador=', repr(actual), 'columnas=', header)
    return header

URL_RE = re.compile(r'https?://[^\s]+|www\.[^\s]+', re.IGNORECASE)

MENTION_RE = re.compile(r'@\w+', re.UNICODE)

REPEAT_RE = re.compile(r'([^\W\d_])\1{3,}', re.UNICODE)

def replace_url(match):
    value = match.group(0)
    core = value.rstrip('.,!?;:)]}')
    return '<url>' + value[len(core):]

def normalize_text(text):
    return REPEAT_RE.sub(lambda match: match.group(1) * 3, MENTION_RE.sub('<user>', URL_RE.sub(replace_url, text)))

verify_csv(DATA_DIR/'age_train.csv',['id','text','age_range'],';',True)
reference_columns = verify_csv(DATA_DIR/'age_submission.csv',['id','age_range'],',',False)
train_raw = pd.read_csv(DATA_DIR/'age_train.csv',sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
assert list(train_raw.columns)==['id','text','age_range']
clean = train_raw.text.fillna('').str.strip().map(html.unescape)
excluded = clean.eq('')|clean.str.contains('#\u00bfNOMBRE',regex=False)
train = train_raw.loc[~excluded].copy()
train['raw_row'] = np.flatnonzero(~excluded.to_numpy())
train['text'] = clean.loc[~excluded]
train = train.reset_index(drop=True)
assert train.id.is_unique and set(train.age_range)==set(CLASSES)
y_all = pd.Categorical(train.age_range,categories=CLASSES).codes.astype(np.int64)

def normalize_a(text):
    return normalize_text(text)

def normalize_b(text):
    return MENTION_RE.sub('<user>',URL_RE.sub(replace_url,text))

def normalize_c(text):
    return normalize_text(text)

def normalize_d(text):
    return normalize_text(text)

ARM_FUNCTIONS = {'A':normalize_a,'B':normalize_b,'C':normalize_c}
arm_texts = {arm:train.text.map(function).to_numpy() for arm,function in ARM_FUNCTIONS.items()}
arm_counts = pd.DataFrame([dict(brazo=arm,solo_url=sum(text.strip()=='<url>' for text in texts),solo_user=sum(text.strip()=='<user>' for text in texts)) for arm,texts in arm_texts.items()])
print('Filas originales/limpias/excluidas:',len(train_raw),len(train),int(excluded.sum()))
print('C conserva caso solo en palabras; caracteres ya lo conservan. D OMITIDO: v6 ya conserva emojis y hashtags.')
display(arm_counts)
assert normalize_a('Hoooooola!!!! @Ana')=='Hooola!!!! <user>'
assert normalize_b('Hoooooola!!!! @Ana')=='Hoooooola!!!! <user>'
assert normalize_a('#Hola \U0001f600')=='#Hola \U0001f600'


age_train.csv BOM= True separador= ';' columnas= ['id', 'text', 'age_range']
age_submission.csv BOM= False separador= ',' columnas= ['id', 'age_range']


Filas originales/limpias/excluidas: 169336 169181 155
C conserva caso solo en palabras; caracteres ya lo conservan. D OMITIDO: v6 ya conserva emojis y hashtags.


,brazo,solo_url,solo_user
0,A,3785,287
1,B,3785,287
2,C,3785,287


In [3]:
group_keys = np.array([text.lower() for text in arm_texts['A']],dtype=object)
group_ids,unique_keys = pd.factorize(group_keys,sort=True)
group_counts = np.zeros((len(unique_keys),6),dtype=np.int64)
np.add.at(group_counts,(group_ids,y_all),1)
group_targets = group_counts.argmax(axis=1)
group_sizes = np.bincount(group_ids,minlength=len(unique_keys))
conflicting_groups = int(((group_counts>0).sum(axis=1)>1).sum())
all_groups = np.arange(len(unique_keys))
size_candidates = []
for fraction in np.round(np.arange(0.180,0.2601,0.001),3):
    dev_candidate,hold_candidate = train_test_split(all_groups,test_size=float(fraction),stratify=group_targets,random_state=SEED)
    rows = int(group_sizes[hold_candidate].sum())
    size_candidates.append(dict(fraccion=float(fraction),filas_holdout=rows,error_filas=abs(rows-v6_holdout_rows),distancia_a_020=abs(float(fraction)-0.2)))
size_table = pd.DataFrame(size_candidates).sort_values(['error_filas','distancia_a_020','fraccion']).reset_index(drop=True)
selected_fraction = float(size_table.iloc[0].fraccion)
dev_groups,holdout_groups = train_test_split(all_groups,test_size=selected_fraction,stratify=group_targets,random_state=SEED)
idx_development = np.flatnonzero(np.isin(group_ids,dev_groups))
idx_holdout = np.flatnonzero(np.isin(group_ids,holdout_groups))
assert not np.intersect1d(dev_groups,holdout_groups).size
assert len(idx_development)+len(idx_holdout)==len(train)
assert not set(group_keys[idx_development])&set(group_keys[idx_holdout])
groups_development = group_ids[idx_development]
y_development = y_all[idx_development]
group_folds = list(StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED).split(dev_groups,group_targets[dev_groups]))
folds = []
for fit_positions,app_positions in group_folds:
    fit_idx = np.flatnonzero(np.isin(groups_development,dev_groups[fit_positions]))
    app_idx = np.flatnonzero(np.isin(groups_development,dev_groups[app_positions]))
    folds.append((fit_idx,app_idx))
leak_rows = []
for arm,texts in arm_texts.items():
    for fold_number,(fit_idx,app_idx) in [(0,(idx_development,idx_holdout))]+[(number,(idx_development[fit_idx],idx_development[app_idx])) for number,(fit_idx,app_idx) in enumerate(folds,1)]:
        fit_keys = {text.lower() for text in texts[fit_idx]}
        app_keys = np.array([text.lower() for text in texts[app_idx]],dtype=object)
        shared = fit_keys&set(app_keys)
        affected = int(np.isin(app_keys,list(shared)).sum())
        leak_rows.append(dict(brazo=arm,pliegue='holdout' if fold_number==0 else str(fold_number),textos_compartidos=len(shared),filas_validacion_afectadas=affected))
        if shared:
            print('DETENER FUGA',arm,fold_number,'ejemplos',sorted(shared)[:10],'ids afectados',train.id.iloc[app_idx[np.isin(app_keys,list(shared))]].tolist())
            raise RuntimeError('Fuga en minusculas: no entrenar ni continuar.')
leak_table = pd.DataFrame(leak_rows)
display(leak_table)
group_hash = hashlib.sha256(json.dumps(group_keys.tolist(),ensure_ascii=False,separators=(',',':')).encode('utf-8')).hexdigest()
with SPLIT_PATH.open('xb') as handle:
    np.savez_compressed(handle,train_idx=idx_development,valid_idx=idx_holdout,raw_train_idx=train.raw_row.to_numpy()[idx_development],raw_valid_idx=train.raw_row.to_numpy()[idx_holdout],retained_raw_idx=train.raw_row.to_numpy(),group_ids=group_ids,train_groups=dev_groups,valid_groups=holdout_groups,seed=np.array(SEED),group_test_size=np.array(selected_fraction),train_sha256=np.array(INPUT_HASHES['age_train.csv']),groups_sha256=np.array(group_hash))
split_table = pd.DataFrame({'particion':['v6 referencia','v7_lower'],'filas_desarrollo':[v6_development_rows,len(idx_development)],'filas_holdout':[v6_holdout_rows,len(idx_holdout)]})
print('Nueva particion:',SPLIT_PATH.name,'fraccion de grupos',selected_fraction,'grupos dev/holdout',len(dev_groups),len(holdout_groups),'grupos contradictorios',conflicting_groups)
print('SHA-256 train:',INPUT_HASHES['age_train.csv'],'| SHA-256 grupos:',group_hash)
print('Hash grupos: JSON UTF-8 compacto de normalizacion A en minusculas, en orden de filas limpias.')
display(split_table)
display(size_table.head(5))


,brazo,pliegue,textos_compartidos,filas_validacion_afectadas
0,A,holdout,0,0
1,A,1,0,0
2,A,2,0,0
3,A,3,0,0
4,A,4,0,0
5,A,5,0,0
6,B,holdout,0,0
7,B,1,0,0
8,B,2,0,0
9,B,3,0,0


Nueva particion: split_idx_v7_lower_seed42.npz fraccion de grupos 0.198 grupos dev/holdout 126309 31184 grupos contradictorios 716
SHA-256 train: 39c6c8270366fe155f3c8dd232e31867bee659af8908121d04b6bd89bd163748 | SHA-256 grupos: f482fe9f4a7495a0050b857b8eda3b3bc9177c31e2ae6f10c9138cfe8df2b1a8
Hash grupos: JSON UTF-8 compacto de normalizacion A en minusculas, en orden de filas limpias.


,particion,filas_desarrollo,filas_holdout
0,v6 referencia,133065,36116
1,v7_lower,132987,36194


,fraccion,filas_holdout,error_filas,distancia_a_020
0,0.198,36194,78,0.002
1,0.197,36031,85,0.003
2,0.199,36352,236,0.001
3,0.196,35873,243,0.004
4,0.200,36515,399,0.000


In [4]:
TOKEN_RE = re.compile(r'<url>|<user>|\w+|[^\w\s]', re.UNICODE)

def tokenize(text, level):
    return list(text) if level == 'char' else TOKEN_RE.findall(text.lower())

def make_encoder(texts, level):
    frequencies = Counter()
    for text in texts:
        frequencies.update(tokenize(text, level))
    cutoff = 1 if level == 'char' else 2
    return {token: chr(index + 256) for index, token in enumerate(sorted(token for token, count in frequencies.items() if count >= cutoff))}

def encode_text(text, level, mapping):
    return ''.join(mapping.get(token, UNK) for token in tokenize(text, level))

def estimate_discounts(counts):
    histogram = Counter(counts.values())
    count1, count2, count3, count4 = [histogram.get(value, 0) for value in [1, 2, 3, 4]]
    factor = count1 / (count1 + 2 * count2) if count1 + 2 * count2 else 0.0
    first = 1 - 2 * factor * count2 / count1 if count1 and count2 else 0.75
    second = 2 - 3 * factor * count3 / count2 if count1 and count2 else 0.75
    third = 3 - 4 * factor * count4 / count3 if count1 and count2 and count3 else 0.75
    values = tuple(float(min(bucket - 1e-6, max(1e-6, value))) for bucket, value in enumerate([first, second, third], 1))
    return values, dict(N1=count1, N2=count2, N3=count3, N4=count4)

def discount_for(count, discounts):
    return discounts[min(count, 3) - 1] if count > 0 else 0.0

class ModifiedKN:
    def __init__(self, max_order, support):
        self.max_order = max_order
        self.support = set(support) | {EOS, UNK}

    def fit(self, sequences):
        highest = max(2, self.max_order)
        self.raw = {order: Counter() for order in range(2, highest + 1)}
        for sequence in sequences:
            padded = BOS * (highest - 1) + sequence + EOS
            for order, counts in self.raw.items():
                counts.update(padded[position - order + 1:position + 1] for position in range(highest - 1, len(padded)))
        self.continuation = {}
        for order in range(1, highest):
            effective = Counter()
            for gram in self.raw[order + 1]:
                effective[gram[1:]] += 1
            self.continuation[order] = effective
        self.base = self.continuation[1]
        self.base_total = sum(self.base.values()) + BASE_EPSILON * len(self.support)
        self.discount_records = []
        def prepare(count_tables, origin):
            discounts_by_order, histories_by_order = {}, {}
            for order, counts in count_tables.items():
                if order == 1:
                    continue
                discounts, histogram = estimate_discounts(counts)
                discounts_by_order[order] = discounts
                totals, removed = Counter(), Counter()
                for gram, count in counts.items():
                    totals[gram[:-1]] += count
                    removed[gram[:-1]] += discount_for(count, discounts)
                histories_by_order[order] = {history: (total, removed[history]) for history, total in totals.items()}
                self.discount_records.append(dict(origen=origin, orden=order, D1=discounts[0], D2=discounts[1], D3=discounts[2], **histogram))
            return discounts_by_order, histories_by_order
        self.raw_discounts, self.raw_hist = prepare(self.raw, 'frecuencias')
        self.cont_discounts, self.cont_hist = prepare(self.continuation, 'continuaciones')
        self.lower_cached = lru_cache(maxsize=200000)(self._lower)
        self.log_cached = lru_cache(maxsize=200000)(self._log_gram)
        return self

    def _lower(self, gram):
        if len(gram) == 1:
            return (self.base.get(gram, 0) + BASE_EPSILON) / self.base_total
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.cont_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.continuation[order].get(gram, 0)
        return (count - discount_for(count, self.cont_discounts[order]) + removed * lower) / total

    def probability(self, gram):
        if len(gram) == 1:
            return self.lower_cached(gram)
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.raw_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.raw[order].get(gram, 0)
        return (count - discount_for(count, self.raw_discounts[order]) + removed * lower) / total

    def _log_gram(self, gram):
        probability = self.probability(gram)
        if not 0 < probability <= 1 + 1e-10:
            raise ArithmeticError((repr(gram), probability))
        return math.log(probability)

    def log_probability(self, sequence, order):
        assert 1 <= order <= self.max_order
        padded = BOS * (order - 1) + sequence + EOS
        grams = Counter(padded[position - order + 1:position + 1] for position in range(order - 1, len(padded)))
        return sum(count * self.log_cached(gram) for gram, count in grams.items())

    def close(self):
        self.lower_cached.cache_clear()
        self.log_cached.cache_clear()

def test_modified_kn():
    values, _ = estimate_discounts(Counter({'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 1, 'f': 2, 'g': 3, 'h': 1}))
    assert len(set(values)) == 3 and all(0 < value < bucket for bucket, value in enumerate(values, 1))
    model = ModifiedKN(4, set('abc')).fit(['ab', 'ab', 'ac', 'bc', 'abc', 'abc', 'abc'])
    assert model.base['a'] == 1 and model.raw[2][BOS + 'a'] == 6
    for order in [1, 2, 3, 4]:
        contexts = set(model.raw_hist.get(order, {})) | {'', 'zz', BOS * (order - 1)}
        for context in contexts:
            history = (BOS * (order - 1) + context)[-(order - 1):] if order > 1 else ''
            assert abs(sum(model.probability(history + symbol) for symbol in model.support) - 1) < 1e-9, (order, repr(history))
        assert math.isfinite(model.log_probability('a' + UNK, order))
    assert model.log_probability('ab', 4) > model.log_probability('cc', 4)
    model.close()
    assert tokenize('Hola HOLA', 'word') == ['hola', 'hola']
    assert tokenize('Hola', 'char')[0] == 'H'
    print('Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.')

def arm_tokens(text,level,arm):
    if arm=='C' and level=='word':
        return TOKEN_RE.findall(text)
    return tokenize(text,level)

def fit_score_level(fit_texts,fit_labels,app_texts,level,order,arm,phase,fold):
    started = time.perf_counter()
    frequencies = Counter()
    for text in fit_texts: frequencies.update(arm_tokens(text,level,arm))
    cutoff = 1 if level=='char' else 2
    mapping = {token:chr(index+256) for index,token in enumerate(sorted(token for token,count in frequencies.items() if count>=cutoff))}
    encoded_fit = [''.join(mapping.get(token,UNK) for token in arm_tokens(text,level,arm)) for text in fit_texts]
    encoded_app = [''.join(mapping.get(token,UNK) for token in arm_tokens(text,level,arm)) for text in app_texts]
    scores = np.zeros((len(app_texts),6))
    for class_index,label in enumerate(CLASSES):
        model = ModifiedKN(order,set(mapping.values())).fit(sequence for sequence,target in zip(encoded_fit,fit_labels) if target==class_index)
        scores[:,class_index] = [model.log_probability(sequence,order) for sequence in encoded_app]
        model.close()
        del model
        gc.collect()
    elapsed = time.perf_counter()-started
    timing_rows.append(dict(fase=phase,pliegue=fold,brazo=arm,nivel=level,segundos=elapsed))
    print(phase,fold,arm,level,'listo',round(elapsed,2),'s',flush=True)
    return scores

test_modified_kn()
assert arm_tokens('Hola HOLA','word','A')==['hola','hola']
assert arm_tokens('Hola HOLA','word','C')==['Hola','HOLA']
assert arm_tokens('Hola','char','A')==arm_tokens('Hola','char','C')
check_texts=['Hola hola','Noticias del dia','Hola amiga']
for level in ['char','word']:
    original_mapping=make_encoder(check_texts,level)
    frequencies=Counter(token for text in check_texts for token in arm_tokens(text,level,'A'))
    cutoff=1 if level=='char' else 2
    arm_mapping={token:chr(index+256) for index,token in enumerate(sorted(token for token,count in frequencies.items() if count>=cutoff))}
    assert original_mapping==arm_mapping
    assert [encode_text(text,level,original_mapping) for text in check_texts]==[''.join(arm_mapping.get(token,UNK) for token in arm_tokens(text,level,'A')) for text in check_texts]
print('Modelo ModifiedKN identico a v6; codificacion A equivalente; C solo cambia caso en palabras.')
timing_rows=[]


Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.
Modelo ModifiedKN identico a v6; codificacion A equivalente; C solo cambia caso en palabras.


In [5]:
cv_started = time.perf_counter()
oof_scores = {arm:np.zeros((len(y_development),6)) for arm in ARM_FUNCTIONS}
fold_id = np.full(len(y_development),-1,dtype=int)
for fold_number,(fit_idx,app_idx) in enumerate(folds,1):
    fit_labels=y_development[fit_idx]
    prior=np.log(np.bincount(fit_labels,minlength=6)/len(fit_labels))
    for arm,texts in arm_texts.items():
        fit_texts=texts[idx_development[fit_idx]]
        app_texts=texts[idx_development[app_idx]]
        assert not {text.lower() for text in fit_texts}&{text.lower() for text in app_texts}
        chars=fit_score_level(fit_texts,fit_labels,app_texts,'char',CHAR_ORDER,arm,'CV',fold_number)
        words=fit_score_level(fit_texts,fit_labels,app_texts,'word',WORD_ORDER,arm,'CV',fold_number)
        oof_scores[arm][app_idx]=WEIGHT*chars+(1-WEIGHT)*words+prior
        del chars,words
        gc.collect()
    fold_id[app_idx]=fold_number
assert (fold_id>0).all()
oof_predictions={arm:scores.argmax(axis=1) for arm,scores in oof_scores.items()}
fold_f1={arm:np.array([f1_score(y_development[app_idx],predictions[app_idx],labels=np.arange(6),average='macro',zero_division=0) for _,app_idx in folds]) for arm,predictions in oof_predictions.items()}
result_rows=[]
paired_rows=[]
for arm in ARM_FUNCTIONS:
    differences=fold_f1[arm]-fold_f1['A']
    result_rows.append(dict(brazo=arm,f1_macro_oof=f1_score(y_development,oof_predictions[arm],labels=np.arange(6),average='macro',zero_division=0),f1_pliegues_media=float(fold_f1[arm].mean()),f1_pliegues_std=float(fold_f1[arm].std(ddof=1)),diferencia_pareada_media=float(differences.mean()),diferencia_pareada_std=float(differences.std(ddof=1))))
    for number,(value,difference) in enumerate(zip(fold_f1[arm],differences),1):
        paired_rows.append(dict(brazo=arm,pliegue=number,f1_macro=value,diferencia_con_A=difference))
results=pd.DataFrame(result_rows)
paired_table=pd.DataFrame(paired_rows)
print('TABLA OOF SIN CALIBRAR; desviaciones muestrales ddof=1')
print(results.to_string(index=False))
display(paired_table)
print('CV segundos',round(time.perf_counter()-cv_started,2))
display(pd.DataFrame(timing_rows))


CV 1 A char listo 6.69 s


CV 1 A word listo 5.64 s


CV 1 B char listo 6.52 s


CV 1 B word listo 5.84 s


CV 1 C char listo 6.58 s


...

CV 5 C word listo 5.55 s


TABLA OOF SIN CALIBRAR; desviaciones muestrales ddof=1
brazo  f1_macro_oof  f1_pliegues_media  f1_pliegues_std  diferencia_pareada_media  diferencia_pareada_std
    A      0.440081           0.440079         0.005308                  0.000000                0.000000
    B      0.439709           0.439704         0.005577                 -0.000375                0.000503
    C      0.436893           0.436895         0.004108                 -0.003184                0.001323


,brazo,pliegue,f1_macro,diferencia_con_A
0,A,1,0.440843,0.000000
1,A,2,0.432485,0.000000
2,A,3,0.440697,0.000000
3,A,4,0.447354,0.000000
4,A,5,0.439014,0.000000
5,B,1,0.440225,-0.000618
6,B,2,0.431630,-0.000856
7,B,3,0.439985,-0.000712
8,B,4,0.447367,0.000012
9,B,5,0.439314,0.000300


CV segundos 181.78


,fase,pliegue,brazo,nivel,segundos
0,CV,1,A,char,6.692057
1,CV,1,A,word,5.636712
2,CV,1,B,char,6.524080
3,CV,1,B,word,5.837800
4,CV,1,C,char,6.581174
5,CV,1,C,word,5.701011
6,CV,2,A,char,6.752731
7,CV,2,A,word,6.078457
8,CV,2,B,char,6.515836
9,CV,2,B,word,5.445824


In [6]:
accepted=results.loc[(results.brazo!='A')&(results.diferencia_pareada_media>0)&(results.diferencia_pareada_media>results.diferencia_pareada_std)].sort_values(['diferencia_pareada_media','brazo'],ascending=[False,True])
PASA=not accepted.empty
selected_arm=str(accepted.iloc[0].brazo) if PASA else None
best_oof_arm=str(results.sort_values(['f1_macro_oof','brazo'],ascending=[False,True]).iloc[0].brazo)
print('PASA' if PASA else 'NO PASA')
print('Regla: diferencia pareada media >0 y >desviacion muestral pareada; brazos aceptados',accepted.brazo.tolist())
print('Brazo elegido para holdout:',selected_arm,'| Mejor OOF descriptivo:',best_oof_arm)
print('Elegir entre brazos introduce optimismo de seleccion. A fue recalculado sobre v7_lower; no se uso un F1 historico como baseline.')
if not PASA:
    print('NO PASA: holdout y CSV se omiten explicitamente; no se creara age_submission_v7_normalizacion.csv.')


NO PASA
Regla: diferencia pareada media >0 y >desviacion muestral pareada; brazos aceptados []
Brazo elegido para holdout: None | Mejor OOF descriptivo: A
Elegir entre brazos introduce optimismo de seleccion. A fue recalculado sobre v7_lower; no se uso un F1 historico como baseline.
NO PASA: holdout y CSV se omiten explicitamente; no se creara age_submission_v7_normalizacion.csv.


In [7]:
if PASA:
    assert 'holdout_f1' not in globals(),'Holdout ya evaluado; no reutilizar.'
    started=time.perf_counter()
    texts=arm_texts[selected_arm]
    fit_texts=texts[idx_development]
    app_texts=texts[idx_holdout]
    assert not {text.lower() for text in fit_texts}&{text.lower() for text in app_texts}
    chars=fit_score_level(fit_texts,y_development,app_texts,'char',CHAR_ORDER,selected_arm,'holdout',0)
    words=fit_score_level(fit_texts,y_development,app_texts,'word',WORD_ORDER,selected_arm,'holdout',0)
    prior=np.log(np.bincount(y_development,minlength=6)/len(y_development))
    holdout_pred=(WEIGHT*chars+(1-WEIGHT)*words+prior).argmax(axis=1)
    y_holdout=y_all[idx_holdout]
    holdout_f1=f1_score(y_holdout,holdout_pred,labels=np.arange(6),average='macro',zero_division=0)
    print('F1 macro holdout v7_lower:',holdout_f1,'| brazo',selected_arm)
    report=classification_report(y_holdout,holdout_pred,labels=np.arange(6),target_names=CLASSES,output_dict=True,zero_division=0)
    display(pd.DataFrame(report).T.loc[CLASSES,['precision','recall','f1-score','support']])
    display(pd.DataFrame(confusion_matrix(y_holdout,holdout_pred,labels=np.arange(6)),index=CLASSES,columns=CLASSES))
    print('No se compara directamente con el holdout de v6: particiones distintas y posible fuga en minusculas en v6.')
    print('Tiempo holdout',round(time.perf_counter()-started,2),'s')
    del chars,words
    gc.collect()
else:
    print('HOLDOUT OMITIDO: NO PASA; no se entreno ni evaluo en holdout.')


HOLDOUT OMITIDO: NO PASA; no se entreno ni evaluo en holdout.


In [8]:
if PASA:
    verify_csv(DATA_DIR/'age_test.csv',['id','text'],';',True)
    test=pd.read_csv(DATA_DIR/'age_test.csv',sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False)
    assert list(test.columns)==['id','text'] and test.id.is_unique and test.id.ne('').all()
    clean_test=test.text.fillna('').str.strip().map(html.unescape)
    app_texts=clean_test.map(ARM_FUNCTIONS[selected_arm]).to_numpy()
    fit_texts=arm_texts[selected_arm]
    chars=fit_score_level(fit_texts,y_all,app_texts,'char',CHAR_ORDER,selected_arm,'final',0)
    words=fit_score_level(fit_texts,y_all,app_texts,'word',WORD_ORDER,selected_arm,'final',0)
    prior=np.log(np.bincount(y_all,minlength=6)/len(y_all))
    predictions=(WEIGHT*chars+(1-WEIGHT)*words+prior).argmax(axis=1)
    submission=pd.DataFrame({'id':test.id,'age_range':CLASSES[predictions]})
    assert list(submission.columns)==reference_columns==['id','age_range']
    assert len(submission)==len(test)==112892 and submission.id.is_unique and submission.id.tolist()==test.id.tolist()
    assert submission.age_range.isin(CLASSES).all()
    with SUB_PATH.open('x',encoding='utf-8',newline='') as handle:
        submission.to_csv(handle,index=False,sep=',')
    with SUB_PATH.open('rb') as handle: prefix=handle.read(64)
    assert not prefix.startswith(b'\xef\xbb\xbf') and prefix.splitlines()[0]==b'id,age_range'
    print('CSV VERIFICADO:',SUB_PATH.name,'112892 filas, IDs en orden, clases validas, coma sin BOM.')
    del chars,words
    gc.collect()
else:
    assert not SUB_PATH.exists()
    print('CSV OMITIDO: NO PASA; no se carga test para prediccion ni se crea salida.')


CSV OMITIDO: NO PASA; no se carga test para prediccion ni se crea salida.


In [9]:
best_predictions=oof_predictions[best_oof_arm]
print('Matriz de confusion del mejor OOF descriptivo:',best_oof_arm)
display(pd.DataFrame(confusion_matrix(y_development,best_predictions,labels=np.arange(6)),index=pd.Index(CLASSES,name='real'),columns=pd.Index(CLASSES,name='predicha')))
oof_report=classification_report(y_development,best_predictions,labels=np.arange(6),target_names=CLASSES,output_dict=True,zero_division=0)
oof_class_table=pd.DataFrame(oof_report).T.loc[CLASSES,['precision','recall','f1-score','support']]
display(oof_class_table)
worst_class=oof_class_table['f1-score'].idxmin()
print('Hipotesis: la confusion entre rangos vecinos y la falta de contenido en textos solo URL/usuario pueden dificultar',worst_class,'; la matriz no demuestra causalidad.')


Matriz de confusion del mejor OOF descriptivo: A


predicha,13-17,18-24,25-34,35-49,50-64,65-xx
real,,,,,,
13-17,10962,5783,2556,1329,1286,876
18-24,4775,8941,3948,1948,1640,1300
25-34,2511,4713,7473,3079,2607,2022
35-49,1283,2452,3756,7554,3988,3135
50-64,680,1235,1990,2714,11689,3715
65-xx,658,1141,1346,1948,3907,12047


,precision,recall,f1-score,support
13-17,0.525277,0.480958,0.502141,22792.0
18-24,0.368473,0.396462,0.381955,22552.0
25-34,0.354692,0.333542,0.343792,22405.0
35-49,0.406741,0.340761,0.370839,22168.0
50-64,0.465382,0.530763,0.495927,22023.0
65-xx,0.521628,0.572386,0.545829,21047.0


Hipotesis: la confusion entre rangos vecinos y la falta de contenido en textos solo URL/usuario pueden dificultar 25-34 ; la matriz no demuestra causalidad.


In [10]:
history_table=pd.DataFrame({'referencia':['v5 calibrado historico','v6 sin calibrar historico','v6 calibrado historico'],'f1_oof_publicado':[0.43847,0.44106,0.44353]})
display(history_table)
display(results)
display(leak_table)
display(split_table)
summary_text=f"""## Resumen final

Resultado: **{'PASA' if PASA else 'NO PASA'}**. Referencia A recalculada: **{float(results.loc[results.brazo=='A','f1_macro_oof'].iloc[0]):.5f}**. Brazo aceptado elegido: **{selected_arm if PASA else 'ninguno'}**. {'Holdout evaluado una sola vez y CSV generado.' if PASA else 'Holdout y CSV omitidos explicitamente.'}

Particion v7_lower: **{len(idx_development)} filas desarrollo / {len(idx_holdout)} holdout**, frente a **{v6_development_rows} / {v6_holdout_rows}** de v6. Fraccion de grupos **{selected_fraction:.3f}**; intersecciones en minusculas **cero** para A, B y C en holdout y los cinco pliegues. El NPZ de v6 quedo intacto y no se usaron sus indices para entrenar.

Cada brazo cambia una sola regla. B conserva repeticiones; C conserva mayusculas en palabras. D se omitio porque v6 conserva emojis y hashtags. No se calibro ningun sesgo. Elegir entre brazos introduce optimismo de seleccion; la regla de media mayor que desviacion no constituye una prueba formal de significacion.

Los valores historicos permitidos de v5/v6 se muestran solo como contexto. El holdout de v6 pudo tener fuga en minusculas: no es comparable directamente con el nuevo. El score publico de Kaggle tampoco es comparable directamente con holdout. Las interpretaciones de la matriz de confusion son hipotesis.

Se guardaron indices, semilla y hashes de train y grupos en **{SPLIT_PATH.name}**. No se cambiaron modelos anteriores, datos, CSV, NPZ existentes ni requirements. Sin commit, push ni envios a Kaggle.
"""
display(Markdown(summary_text))
assert INPUT_HASHES=={name:hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest() for name in INPUT_HASHES}
assert PROTECTED_HASHES=={name:hashlib.sha256((OUT_DIR/name).read_bytes()).hexdigest() for name in PROTECTED_HASHES}
print('HASHES DE ARCHIVOS PREEXISTENTES PROTEGIDOS: SIN CAMBIOS.')


,referencia,f1_oof_publicado
0,v5 calibrado historico,0.43847
1,v6 sin calibrar historico,0.44106
2,v6 calibrado historico,0.44353


,brazo,f1_macro_oof,f1_pliegues_media,f1_pliegues_std,diferencia_pareada_media,diferencia_pareada_std
0,A,0.440081,0.440079,0.005308,0.000000,0.000000
1,B,0.439709,0.439704,0.005577,-0.000375,0.000503
2,C,0.436893,0.436895,0.004108,-0.003184,0.001323


,brazo,pliegue,textos_compartidos,filas_validacion_afectadas
0,A,holdout,0,0
1,A,1,0,0
2,A,2,0,0
3,A,3,0,0
4,A,4,0,0
5,A,5,0,0
6,B,holdout,0,0
7,B,1,0,0
8,B,2,0,0
9,B,3,0,0


,particion,filas_desarrollo,filas_holdout
0,v6 referencia,133065,36116
1,v7_lower,132987,36194


## Resumen final

Resultado: **NO PASA**. Referencia A recalculada: **0.44008**. Brazo aceptado elegido: **ninguno**. Holdout y CSV omitidos explicitamente.

Particion v7_lower: **132987 filas desarrollo / 36194 holdout**, frente a **133065 / 36116** de v6. Fraccion de grupos **0.198**; intersecciones en minusculas **cero** para A, B y C en holdout y los cinco pliegues. El NPZ de v6 quedo intacto y no se usaron sus indices para entrenar.

Cada brazo cambia una sola regla. B conserva repeticiones; C conserva mayusculas en palabras. D se omitio porque v6 conserva emojis y hashtags. No se calibro ningun sesgo. Elegir entre brazos introduce optimismo de seleccion; la regla de media mayor que desviacion no constituye una prueba formal de significacion.

Los valores historicos permitidos de v5/v6 se muestran solo como contexto. El holdout de v6 pudo tener fuga en minusculas: no es comparable directamente con el nuevo. El score publico de Kaggle tampoco es comparable directamente con holdout. Las interpretaciones de la matriz de confusion son hipotesis.

Se guardaron indices, semilla y hashes de train y grupos en **split_idx_v7_lower_seed42.npz**. No se cambiaron modelos anteriores, datos, CSV, NPZ existentes ni requirements. Sin commit, push ni envios a Kaggle.


HASHES DE ARCHIVOS PREEXISTENTES PROTEGIDOS: SIN CAMBIOS.
